# 무선청소기 브랜드 네트워크

의류관리기 네트워크 분석 방법을 무선청소기에 맞춰 정리한 공개용 코드. 당시 실행본은 보존되지 않았음.


In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager
import networkx as nx
from collections import Counter

# 데이터 로드

In [ ]:
# 형태소 처리된 비공개 분석 입력 위치
file_path = os.environ['LG_VOC_DATA_DIR'] + os.sep

vacuum = pd.read_csv(file_path + '무선스틱청소기_수정.csv', encoding='utf-8-sig')

## 불용어 처리

In [ ]:
### 불용어 처리 리스트
stop_words = {'이다', '에서', '후', '번', '일', '랑', '보다', '를', '와', '내', '과', '되다', '전자서비스센터',
              '시', '때', '데', '을', '는', '가', '이', '개', '때문', '꼭', '경우', '은', '에도',
              '되어다', '수', '의', '없다', '나', '까지', '이제', '같다', '뿐', '해주다', '않다',
              '다이렉트결혼준비', '에', '전', '들', '중', '다', '기', '그렇다', '니', '도', '등', '있다',
              '분', '더', '정말', '습', '게', '하다', '적', '아니다', '매', '더욱', '슈', '인', '안', '해', 'S',
              '만', '바로', '로', '한', 'CB', '처럼', '니다', '그', '오브', '그리고', '통해', '점', '제', '이나',
              '한번', '글', '체험단', '요', '으로', '저희', '거', '하고', '것', '부터', '에는', '기다', '저',
              '블랙틴트미러', '그래서', '특히', '예요', '또한', '많이', '없이', '호텔', '수업', '교시', '출판',
              '서점', '이벤트', '발매', '돼다', '보시', '이라', '참고', '오다', '맞다', '같이', '근데', '그냥', '약간',
              '다음', '여기', '조금', '수가', '하나', '대해', '어떻다', '싶다', '이라고', '너무', '정도', '자주', '이렇게',
              '인데', '이기', '가장', '하지만', '해당', '한다는', '사하', '가장', '는걸', '아주', '고', '\n', '\xa0',
              ' ', '레몬테라스카페', '즉', '다음', '동영상', '취소', '재생', '속도', '음소거', '음소거설정', '실시간', '설정',
              '실시간설정', '다시시도',
              '골프', '레슨', '도움말', '도움말상태', '라이센스', '고화질', '영상', '해상도', '문제', '발생',
              '해상도', '자막', '안함', '초', '광고', '공유', '전체화면', '자막안함옵션',
              '글자', '크기배경색', '도움말', '상태', '배경색기본', '골프연습장', '회원님',
              '골프레슨', '죄송하다', '문제발생', '가능한영상', '자막안함', '글자크기',
              '범례부동산거리', '전체화면', '크기배경색','옵션', '기본자동옵션', '시가총액', '억원', '상승',
              '차량용', '차량', '용', '원', '현재', '동시', '군', '구시', '거리', '읍', '면', '도국', '지도',
              '범례', '부동산', '거리', '데이터', '다시시도', '존', '인', '피트', '회원님', '들다', '기본옵션', '오후', '우리집',
              '년', '집', '및', '준비물', '다시', '시도', '로켓', '배송'
              }

In [ ]:
def remove_stopwords_from_df(df, columns, stop_words):
    # 불용어 처리 함수
    def remove_stopwords(text, stop_words):
        if not isinstance(text, str):
            return ''

        # 띄어쓰기를 기준으로 단어 분리
        tokens = text.split()

        # 불용어 제거
        filtered_tokens = [word for word in tokens if word not in stop_words]

        # 단어를 다시 문장으로 합치기
        return ' '.join(filtered_tokens)

    # 지정된 열에 대해 불용어 처리
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: remove_stopwords(x, stop_words))

    return df

vacuum = remove_stopwords_from_df(vacuum, ['noun', 'verb_adj', 'nva', 'token'], stop_words)

## 연도/월별 열 추가

In [ ]:
# year와 month 열 추가 함수
def add_year_month_columns(df):
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month

add_year_month_columns(vacuum)

# 브랜드별 데이터 나누기

* 브랜드 키워드가 제목이나 본문에 들어간 게시글을 브랜드별로 나눔
* 두 브랜드 이상에 걸린 게시글(link 중복)은 제외

In [ ]:
# 무선청소기 브랜드별 키워드
vacuum_lg_keyword = ['코드제로', 'LG', '엘지', 'lg무선청소기', 'lg무선스틱청소기', '엘지무선청소기', '엘지무선스틱청소기',
                     'lg코드제로a9', 'lg코드제로', '코드제로a9', '엘지코드제로a9']
vacuum_samsung_keyword = ['삼성', '삼성무선청소기', '삼성무선스틱청소기', '삼성제트청정스테이션', '삼성비스포크제트',
                          '비스포크제트', '비스포크슬림', '비스포크슬림삼성']
vacuum_dyson_keyword = ['다이슨', '다이슨무선청소기', '다이슨무선스틱청소기', '디지털슬림', '디지털슬림싸이클론v10',
                        '다이슨청소기디텍트', '다이슨디텍트v15', '디텍트v12', '디텍트슬림']


def filter_data(df, keywords):
    return df[
        df['title'].str.contains('|'.join(keywords), case=False, na=False) |
        df['content'].str.contains('|'.join(keywords), case=False, na=False)
    ]


vacuum_lg = filter_data(vacuum, vacuum_lg_keyword)
vacuum_samsung = filter_data(vacuum, vacuum_samsung_keyword)
vacuum_dyson = filter_data(vacuum, vacuum_dyson_keyword)

In [ ]:
## 중복되는 행 제거
# 1. 모든 데이터프레임의 "link"를 하나로 합치기
all_links = pd.concat([vacuum_lg['link'], vacuum_samsung['link'], vacuum_dyson['link']])

# 2. 두 번 이상 등장한 "link" 추출
common_links = all_links.value_counts()[all_links.value_counts() > 1].index

# 3. 각 데이터프레임에서 중복 "link" 제거
vacuum_lg = vacuum_lg[~vacuum_lg['link'].isin(common_links)]
vacuum_samsung = vacuum_samsung[~vacuum_samsung['link'].isin(common_links)]
vacuum_dyson = vacuum_dyson[~vacuum_dyson['link'].isin(common_links)]

print(vacuum_lg.shape)
print(vacuum_samsung.shape)
print(vacuum_dyson.shape)

# 브랜드별 전체 빈도: 연관어 Top 50

In [ ]:
from collections import Counter

def count_words(text_series, year_series, top_n=None, year1=None, year2=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 전체 텍스트 합치기
    all_text = ' '.join(valid_data['text'])

    # 단어 빈도 계산
    word_counts = Counter(all_text.split())
    total_word_count = sum(word_counts.values()) or 1  # 분모 0 방지

    # 단어 비율 계산
    words_list, freqs = zip(*word_counts.most_common(top_n)) if word_counts else ([], [])
    ratios = [freq / total_word_count for freq in freqs]

    # transition 계산
    transitions = []
    if year1 is not None and year2 is not None:
        # 연도별 데이터 필터링
        year1_texts = valid_data[valid_data['year'] == year1]['text'].dropna().astype(str)
        year2_texts = valid_data[valid_data['year'] == year2]['text'].dropna().astype(str)

        year1_words = ' '.join(year1_texts).split()
        year2_words = ' '.join(year2_texts).split()

        year1_counts = Counter(year1_words)
        year2_counts = Counter(year2_words)

        total1 = sum(year1_counts.values()) or 1  # 분모 0 방지
        total2 = sum(year2_counts.values()) or 1

        for word in words_list:
            count1 = year1_counts.get(word, 0)
            count2 = year2_counts.get(word, 0)

            # transition 계산: 변화율 (year2 빈도 - year1 빈도) / year1 빈도 * 100
            if count1 > 0:
                transition_rate = ((count2/total2 - count1/total1) / (count1/total1)) * 100
            else:
                transition_rate = 0  # year1의 빈도가 0이면 변화율을 0으로 설정

            transitions.append(transition_rate)
    else:
        transitions = [0] * len(words_list)  # 연도별 데이터가 없는 경우 0으로 채움

    # 결과 데이터프레임 생성
    result_df = pd.DataFrame({
        'word': words_list,
        'freq': freqs,
        'ratio': ratios,
        'transition': transitions
    })

    # 빈도(freq) 기준으로 순위를 매기고 rank 열 추가
    result_df['rank'] = result_df['freq'].rank(method='dense', ascending=False).astype(int)

    # rank 열을 가장 왼쪽으로 이동
    result_df = result_df[['rank', 'word', 'freq', 'ratio', 'transition']]

    return result_df

In [ ]:
lg_total_counts = count_words(text_series=vacuum_lg['nva'], year_series=vacuum_lg['year'], top_n=50)
samsung_total_counts = count_words(text_series=vacuum_samsung['nva'], year_series=vacuum_samsung['year'], top_n=50)
dyson_total_counts = count_words(text_series=vacuum_dyson['nva'], year_series=vacuum_dyson['year'], top_n=50)

lg_total_counts.head(10)

## csv 저장

In [ ]:
# CSV로 저장하기
brand_path = os.path.join("results", "브랜드별분석") + os.sep
os.makedirs(brand_path, exist_ok=True)

lg_total_counts.to_csv(brand_path + 'vacuum_lg_total_counts.csv', index=False, encoding='utf-8-sig')
samsung_total_counts.to_csv(brand_path + 'vacuum_samsung_total_counts.csv', index=False, encoding='utf-8-sig')
dyson_total_counts.to_csv(brand_path + 'vacuum_dyson_total_counts.csv', index=False, encoding='utf-8-sig')

# 네트워크 분석

* 브랜드 노드와 각 브랜드의 연관어 Top 50을 연결
* 여러 브랜드의 Top 50에 함께 든 단어는 해당 브랜드 노드 모두와 연결됨

In [ ]:
# word_data1 : LG Top 50
# word_data2 : 삼성 Top 50
# word_data3 : 다이슨 Top 50

word_data1 = lg_total_counts[['word', 'freq']].rename(columns={'word': '단어', 'freq': '빈도'})
word_data1['brand'] = 'vacuum_lg'

word_data2 = samsung_total_counts[['word', 'freq']].rename(columns={'word': '단어', 'freq': '빈도'})
word_data2['brand'] = 'vacuum_samsung'

word_data3 = dyson_total_counts[['word', 'freq']].rename(columns={'word': '단어', 'freq': '빈도'})
word_data3['brand'] = 'vacuum_dyson'

dataframe = pd.concat([word_data1, word_data2, word_data3], ignore_index=True)
dataframe

In [ ]:
# 한글 폰트 설정
font_path = os.environ.get('LG_VOC_FONT_PATH')
font_prop = font_manager.FontProperties(fname=font_path) if font_path else font_manager.FontProperties()
plt.rcParams['axes.unicode_minus'] = False  # 한글 깨짐 방지

In [ ]:
## 네트워크 그래프

# 브랜드 노드 이름 → 그림에 표시할 이름
brand_labels = {'vacuum_lg': 'LG', 'vacuum_samsung': '삼성', 'vacuum_dyson': '다이슨'}
brand_colors = {'vacuum_lg': 'lightcoral', 'vacuum_samsung': 'cornflowerblue', 'vacuum_dyson': 'lightgreen'}

g = nx.from_pandas_edgelist(dataframe, source='단어', target='brand')
pos = nx.spring_layout(g, k=0.7, iterations=200, scale=2, seed=42)

color_map_node = []
node_size_map = []
for node in g:
    if node in brand_colors:
        color_map_node.append(brand_colors[node])
        node_size_map.append(2000)
    else:
        color_map_node.append('white')
        node_size_map.append(1000)

color_map_edge = []
for edges in g.edges():
    for brand, color in brand_colors.items():
        if brand in edges:
            color_map_edge.append(color)

plt.figure(figsize=(20, 17))

nx.draw_networkx_nodes(g, pos, node_size=node_size_map, node_color=color_map_node)
nx.draw_networkx_edges(g, pos, width=1, edge_color=color_map_edge)

# 노드 라벨 추가 (한글 폰트 적용)
labels = {node: brand_labels.get(node, str(node)) for node in g.nodes()}  # 노드 이름 매핑
for key, (x, y) in pos.items():
    plt.text(x, y, labels[key], fontproperties=font_manager.FontProperties(fname=font_path, size=16) if font_path else font_manager.FontProperties(size=16), ha='center', va='center')

plt.axis('off')
plt.title('무선스틱청소기 브랜드별 연관어 Top 50 네트워크', fontproperties=font_manager.FontProperties(fname=font_path, size=20) if font_path else font_manager.FontProperties(size=20), y=-0.05)
plt.savefig(brand_path + 'vacuum_brand_network.png', dpi=300, bbox_inches='tight')
plt.show()

# 브랜드별 연도별 빈도

In [ ]:
import pandas as pd
from collections import Counter

def count_words_by_year(text_series, year_series, year_range, top_n=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 연도 범위 필터링
    valid_data = valid_data[valid_data['year'].isin(year_range)]
    unique_years = sorted(valid_data['year'].unique())

    # 결과 저장용 리스트 초기화
    result_data = []

    # 이전 연도 단어 빈도 초기화
    prev_year_counts = Counter()

    for i, year in enumerate(unique_years):
        # 현재 연도의 텍스트 데이터
        current_texts = valid_data[valid_data['year'] == year]['text']
        all_text = ' '.join(current_texts)

        # 단어 빈도 계산
        word_counts = Counter(all_text.split())
        total_word_count = sum(word_counts.values()) or 1  # 분모 0 방지

        # 단어 비율 계산 및 transition 계산
        words_list, freqs = zip(*word_counts.most_common(top_n)) if word_counts else ([], [])
        ratios = [freq / total_word_count for freq in freqs]

        transitions = []
        if i == 0:  # 첫 번째 연도는 변화율 계산하지 않음
            transitions = [0] * len(words_list)
        else:
            for word in words_list:
                prev_freq = prev_year_counts.get(word, 0)
                prev_total = sum(prev_year_counts.values()) or 1
                prev_ratio = prev_freq / prev_total
                current_ratio = word_counts[word] / total_word_count

                # 변화율 계산
                transition = (current_ratio - prev_ratio) / prev_ratio * 100 if prev_ratio > 0 else 0
                transitions.append(transition)

        # 현재 연도의 결과를 리스트에 저장
        for word, freq, ratio, transition in zip(words_list, freqs, ratios, transitions):
            result_data.append([word, year, freq, ratio, transition])

        # 현재 연도의 단어 빈도를 이전 연도로 설정
        prev_year_counts = word_counts

    # 결과 데이터를 데이터프레임으로 변환
    result_df = pd.DataFrame(result_data, columns=['word', 'year', 'freq', 'ratio', 'transition'])

    # 같은 단어를 기준으로 연도별 빈도와 비율 정렬
    all_words = sorted(result_df['word'].unique())
    final_result = pd.DataFrame({'word': all_words})
    for year in year_range:
        year_data = result_df[result_df['year'] == year][['word', 'freq', 'ratio', 'transition']]
        year_data = year_data.rename(columns={
            'freq': f'freq_{year}',
            'ratio': f'ratio_{year}',
            'transition': f'transition_{year}'
        })
        final_result = final_result.merge(year_data, on='word', how='left')
        final_result[f'word_{year}'] = final_result['word'].where(final_result[f'freq_{year}'].notna())

    # 전체 기간 빈도를 합산해 순위 계산
    freq_columns = [f'freq_{year}' for year in year_range]
    final_result.insert(0, 'rank', final_result[freq_columns].fillna(0).sum(axis=1).rank(method='dense', ascending=False).astype(int))
    final_result = final_result.sort_values('rank').reset_index(drop=True)
    return final_result

In [ ]:
year_range = [2022, 2023, 2024]

lg_total_counts_years = count_words_by_year(text_series=vacuum_lg['nva'], year_series=vacuum_lg['year'], year_range=year_range, top_n=50)
samsung_total_counts_years = count_words_by_year(text_series=vacuum_samsung['nva'], year_series=vacuum_samsung['year'], year_range=year_range, top_n=50)
dyson_total_counts_years = count_words_by_year(text_series=vacuum_dyson['nva'], year_series=vacuum_dyson['year'], year_range=year_range, top_n=50)

# CSV로 저장하기
lg_total_counts_years.to_csv(brand_path + 'vacuum_lg_total_counts_years.csv', index=False, encoding='utf-8-sig')
samsung_total_counts_years.to_csv(brand_path + 'vacuum_samsung_total_counts_years.csv', index=False, encoding='utf-8-sig')
dyson_total_counts_years.to_csv(brand_path + 'vacuum_dyson_total_counts_years.csv', index=False, encoding='utf-8-sig')

## 증감 check!

### 함수

In [ ]:
# 1. 2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어를 확인
def consistently_increasing_words(df):
    increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        increasing_words.append(word)
    
    return increasing_words


# 1. 2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어를 확인
def consistently_decreasing_words(df):
    decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        decreasing_words.append(word)
    
    return decreasing_words


# 1. 2022 -> 2023에는 감소, 2023 -> 2024에는 transition 값이 증가한 단어를 확인
# 2. 2022에는 존재하는 단어지만 2023에는 사라진 단어. 2024에 다시 등장 & transition 값이 증가(양수)한 단어
def recently_increasing_words(df):
    recently_increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)
    
    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 양수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 > 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    return recently_increasing_words


# 1. 2022 -> 2023에는 증가, 2023 -> 2024에는 transition 값이 감소한 단어를 확인
def recently_decreasing_words(df):
    recently_decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 음수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 < 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    return recently_decreasing_words


# 1. 2024년 상위 목록에 새로 등장한 단어를 확인
def recently_appeared_words(df):
    recently_appeared_words = []
                          
    # word_2022, word_2023에는 없지만 word_2024에만 존재하는 단어 확인
    for word in df['word_2024']:
        if pd.isna(word):
            continue
        if word not in df['word_2022'].values and word not in df['word_2023'].values:
            recently_appeared_words.append(word)


    return recently_appeared_words



### 실행

In [ ]:
brand_counts_years = {'LG': lg_total_counts_years, '삼성': samsung_total_counts_years, '다이슨': dyson_total_counts_years}

for brand, df in brand_counts_years.items():
    print(f"<{brand}>")
    print("2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어:")
    print(consistently_increasing_words(df))
    print("2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어:")
    print(consistently_decreasing_words(df))
    print("2023 -> 2024 transition 값이 증가한 단어:")
    print(recently_increasing_words(df))
    print("2023 -> 2024 transition 값이 감소한 단어:")
    print(recently_decreasing_words(df))
    print("2024년 상위 목록에 새로 등장한 단어:")
    print(recently_appeared_words(df))
    print()

# 브랜드 핵심 키워드

* 네트워크에서 해당 브랜드에만 연결된 단어와, 최근 언급 비중이 증가한 단어의 교집합
* 최근 증가: 22 → 23 → 24 증가, 23 → 24 증가, 24 신규 등장
* 브랜드명·모델명 키워드 제외

In [ ]:
# 네트워크에서 해당 브랜드 노드에만 연결된 단어
def brand_only_words(dataframe, brand):
    word_brand_count = dataframe.groupby('단어')['brand'].nunique()
    only_words = word_brand_count[word_brand_count == 1].index
    return dataframe[(dataframe['brand'] == brand) & (dataframe['단어'].isin(only_words))]['단어'].tolist()


# 최근 언급 비중이 증가한 단어
def recently_rising_words(df):
    return set(consistently_increasing_words(df) + recently_increasing_words(df) + recently_appeared_words(df))


# 브랜드명·모델명 키워드
brand_name_words = ['LG', '엘지', '삼성', '비스포크', '비스포크제트', '제트', '다이슨', '디텍트', '슬림']

brand_nodes = {'LG': 'vacuum_lg', '삼성': 'vacuum_samsung', '다이슨': 'vacuum_dyson'}

brand_core_keywords = {}
for brand, df in brand_counts_years.items():
    rising = recently_rising_words(df)
    brand_core_keywords[brand] = [word for word in brand_only_words(dataframe, brand_nodes[brand])
                                  if word in rising and word not in brand_name_words]
    print(f"<{brand} 브랜드 핵심 키워드>")
    print(brand_core_keywords[brand])

# 핵심 키워드 연관 키워드

In [ ]:
import pandas as pd

# 연관 단어 함수
def coocc_word(sentences, keyword, window=10):
    coocc = dict()  # 단어 빈도를 저장할 사전
    for sent in sentences:
        word_list = sent.split(' ')  # 문장을 단어 리스트로 분리
        length = len(word_list)
        for i, word in enumerate(word_list):
            if word == keyword:  # 현재 단어와 키워드가 일치하면,
                for j in range(max(i - window, 0), min(i + window + 1, length)):  # 주변 단어 탐색
                    if word_list[j] in coocc:
                        coocc[word_list[j]] += 1
                    else:
                        coocc[word_list[j]] = 1
    df_occ = pd.Series(coocc, name='occ').sort_values(ascending=False)
    
    # keyword를 포함한 단어 제거
    df_occ = df_occ[df_occ.index != keyword]
    
    return df_occ

def generate_column_order(years):
    base_columns = ['rank']
    
    for year in years:
        base_columns.append(f'word_{year}')
        base_columns.append(f'freq_{year}')
        base_columns.append(f'ratio_{year}')
        base_columns.append(f'transition_{year}')
    
    return base_columns

# 연관 단어 결과로 rank, word, freq, ratio 데이터프레임 생성 함수 (transition 수정)
def coocc_word_with_rank(text_series, year_series, keyword, years, window=10, top_n=None, path=None, df_name=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 결과 저장용 딕셔너리 초기화
    result_dict = {}

    prev_year_counts = {}  # 이전 연도의 단어 빈도를 저장할 딕셔너리

    for i, year in enumerate(years):
        # 현재 연도의 텍스트 데이터
        current_texts = valid_data[valid_data['year'] == year]['text']
        # 게시글 사이의 단어가 연관어로 집계되지 않도록 행별로 계산
        sentences = current_texts.dropna().astype(str).tolist()
        df_occ = coocc_word(sentences, keyword, window)

        # 단어 빈도 계산
        word_counts = df_occ.to_dict()  # 현재 연도 단어 빈도를 딕셔너리로 저장
        words_list = list(word_counts.keys())
        freqs = list(word_counts.values())

        total_word_count = sum(freqs) or 1  # 분모 0 방지

        # 단어 비율 계산
        ratios = [freq / total_word_count for freq in freqs]

        # transition 계산 (이전 연도 대비 변화율)
        transitions = []
        if i == 0:  # 첫 번째 연도는 변화율 계산하지 않음
            transitions = [0] * len(words_list)
        else:
            for word in words_list:
                prev_freq = prev_year_counts.get(word, 0)
                prev_total = sum(prev_year_counts.values()) or 1
                current_ratio = word_counts[word] / total_word_count
                prev_ratio = prev_freq / prev_total

                # 변화율 계산
                transition = (current_ratio - prev_ratio) / prev_ratio * 100 if prev_ratio > 0 else current_ratio * 100
                transitions.append(transition)

        # 현재 연도의 결과를 딕셔너리에 저장
        result_dict[f'word_{year}'] = words_list
        result_dict[f'freq_{year}'] = freqs
        result_dict[f'ratio_{year}'] = ratios
        result_dict[f'transition_{year}'] = transitions

        # 현재 연도의 단어 빈도를 이전 연도로 저장
        prev_year_counts = word_counts

    # 연도별 결과를 같은 연관 단어 기준으로 정렬
    all_words = sorted(set().union(*[set(result_dict[f'word_{year}']) for year in years]))
    final_result = pd.DataFrame({'word': all_words})
    for year in years:
        year_data = pd.DataFrame({
            'word': result_dict[f'word_{year}'],
            f'freq_{year}': result_dict[f'freq_{year}'],
            f'ratio_{year}': result_dict[f'ratio_{year}'],
            f'transition_{year}': result_dict[f'transition_{year}']
        })
        final_result = final_result.merge(year_data, on='word', how='left')
        final_result[f'word_{year}'] = final_result['word'].where(final_result[f'freq_{year}'].notna())

    # 전체 기간 빈도 기준 순위
    freq_cols = [f'freq_{year}' for year in years]
    final_result.insert(0, 'rank', final_result[freq_cols].fillna(0).sum(axis=1).rank(method='dense', ascending=False).astype(int))
    final_result = final_result.sort_values('rank').reset_index(drop=True)

    # top_n이 지정되면 상위 n개의 결과만 선택
    if top_n:
        final_result = final_result.head(top_n)

    # 결과를 CSV로 저장
    if path:
        os.makedirs(path, exist_ok=True)
        column_name = text_series.name or 'text'
        filename = f"{df_name}_{column_name}_{keyword}.csv"
        filepath = os.path.join(path, filename)
        final_result.to_csv(filepath, index=False, encoding="utf-8-sig")
        print(f"Results saved to {filepath}")

    return final_result


In [ ]:
# 경로
coocc_brand_path = brand_path + '연관키워드/'
os.makedirs(coocc_brand_path, exist_ok=True)

brand_data = {'LG': (vacuum_lg, 'vacuum_lg'), '삼성': (vacuum_samsung, 'vacuum_samsung'), '다이슨': (vacuum_dyson, 'vacuum_dyson')}

for brand, keywords in brand_core_keywords.items():
    df_brand, df_name = brand_data[brand]
    for keyword in keywords:
        df = coocc_word_with_rank(df_brand['nva'], df_brand['year'], keyword=keyword, years=[2022, 2023, 2024],
                                  top_n=21, window=1, path=coocc_brand_path, df_name=df_name)

        print(f"<{brand} - {keyword}>")
        print("2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어:")
        print(consistently_increasing_words(df))
        print("2023 -> 2024 transition 값이 증가한 단어:")
        print(recently_increasing_words(df))
        print("2024년 상위 목록에 새로 등장한 단어:")
        print(recently_appeared_words(df))
        print()